# 06 — Build Hybrid Index E5 để Deploy

Xuất model, embedding 768 chiều, BM25 index, metadata, Supabase import và file ZIP.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
%%capture
!pip install bm25s

In [4]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib, json, shutil, zipfile
import bm25s, numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer

ROOT = Path("/content/drive/MyDrive/UIT_Legal_System")
PROCESSED = ROOT/"Dataset/processed"
TRAINED = ROOT/"Source Code/Embedding Model/multilingual_e5_base_finetuned_ver2/checkpoints/checkpoint-484"
EVAL = ROOT/"Source Code/Embedding Model/retriever_evaluation_e5/retriever_winner_config.json"
DEPLOY = ROOT/"artifacts/deploy/hybrid_retriever_e5"
DEPLOY.mkdir(parents=True, exist_ok=True)

CORPUS = pd.read_parquet(PROCESSED/"corpus.parquet").reset_index(drop=True)
with EVAL.open("r", encoding="utf-8") as f:
    WINNER = json.load(f)

FIELD = WINNER.get("corpus_field","embedding_text")
RRF_K = WINNER.get("rrf_k") or 60
MODEL_DIR = DEPLOY/"dense_model"
BM25_DIR = DEPLOY/"bm25_index"

In [5]:
if MODEL_DIR.exists(): shutil.rmtree(MODEL_DIR)
shutil.copytree(TRAINED, MODEL_DIR)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
model = SentenceTransformer(str(MODEL_DIR), device=device, model_kwargs={"torch_dtype":dtype})
model.max_seq_length = 512

def fq(x): return f"query: {str(x).strip()}"
def fp(x): return f"passage: {str(x).strip()}"

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [6]:
passages = [fp(x) for x in CORPUS[FIELD].fillna("").astype(str)]
emb = model.encode(
    passages,
    batch_size=32,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
).astype(np.float32)

if emb.shape[1] != 768:
    raise RuntimeError(f"Expected 768 dimensions, got {emb.shape[1]}")

np.save(DEPLOY/"corpus_embeddings.npy", emb)
print(emb.shape)

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

(1045, 768)


In [7]:
meta = CORPUS[["passage_id","title","document","article","content","embedding_text","content_hash"]].copy()
meta.insert(0,"embedding_index",np.arange(len(meta),dtype=np.int64))
meta["embedding_dimension"] = 768
meta.to_parquet(DEPLOY/"passage_metadata.parquet",index=False)

with (DEPLOY/"passage_metadata.jsonl").open("w",encoding="utf-8") as f:
    for r in meta.to_dict(orient="records"):
        f.write(json.dumps(r,ensure_ascii=False)+"\n")

In [8]:
if BM25_DIR.exists(): shutil.rmtree(BM25_DIR)
texts = CORPUS[FIELD].fillna("").astype(str).tolist()
tokens = bm25s.tokenize(texts,stopwords=None,stemmer=None,show_progress=True)
bm25 = bm25s.BM25(method="lucene")
bm25.index(tokens,show_progress=True)
bm25.save(str(BM25_DIR),corpus=None)

with (DEPLOY/"bm25_corpus.jsonl").open("w",encoding="utf-8") as f:
    for i,row in CORPUS.iterrows():
        f.write(json.dumps({"bm25_index":int(i),"passage_id":str(row["passage_id"]),"text":str(row[FIELD])},ensure_ascii=False)+"\n")

Split strings:   0%|          | 0/1045 [00:00<?, ?it/s]

DEBUG:bm25s:Building index from IDs objects


BM25S Count Tokens:   0%|          | 0/1045 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/1045 [00:00<?, ?it/s]

In [9]:
supabase = meta[["passage_id","title","document","article","content","embedding_text","content_hash","embedding_index"]].copy()
supabase["embedding"] = [v.astype(float).tolist() for v in emb]
supabase["metadata"] = [
    {"title":r["title"],"document":r["document"],"article":r["article"],"content_hash":r["content_hash"],"embedding_index":int(r["embedding_index"])}
    for r in supabase.to_dict(orient="records")
]
supabase.to_parquet(DEPLOY/"supabase_import.parquet",index=False)

with (DEPLOY/"supabase_import.jsonl").open("w",encoding="utf-8") as f:
    for r in supabase.to_dict(orient="records"):
        f.write(json.dumps(r,ensure_ascii=False)+"\n")

In [10]:
config = {
    "schema_version":"1.0",
    "retriever_type":"hybrid_dense_bm25_rrf",
    "model_key":"multilingual_e5_base_finetuned",
    "dense_model_path":"dense_model",
    "corpus_embeddings_path":"corpus_embeddings.npy",
    "passage_metadata_path":"passage_metadata.parquet",
    "bm25_index_path":"bm25_index",
    "bm25_corpus_path":"bm25_corpus.jsonl",
    "supabase_import_path":"supabase_import.parquet",
    "query_instruction":"",
    "query_format":"query: {question}",
    "passage_format":"passage: {passage}",
    "corpus_field":FIELD,
    "max_query_length":128,
    "max_passage_length":512,
    "embedding_dimension":768,
    "normalize_embeddings":True,
    "similarity":"cosine",
    "dense_top_k":30,
    "bm25_top_k":30,
    "final_top_k":8,
    "fusion":{"method":"reciprocal_rank_fusion","rrf_k":int(RRF_K)},
    "bm25":{"library":"bm25s","method":"lucene","stopwords":None,"stemmer":None},
}
with (DEPLOY/"retriever_config.json").open("w",encoding="utf-8") as f:
    json.dump(config,f,ensure_ascii=False,indent=2)

In [11]:
sql = '''create extension if not exists vector with schema extensions;

create table if not exists public.regulation_chunks (
  id bigint generated by default as identity primary key,
  passage_id text not null unique,
  title text not null default '',
  document text not null default '',
  article text not null default '',
  content text not null,
  embedding_text text not null default '',
  content_hash text not null default '',
  embedding extensions.vector(768) not null,
  metadata jsonb not null default '{}'::jsonb,
  created_at timestamptz not null default now()
);

create index if not exists regulation_chunks_embedding_hnsw_idx
on public.regulation_chunks
using hnsw (embedding extensions.vector_cosine_ops);

create or replace function public.match_regulation_chunks(
  query_embedding extensions.vector(768),
  match_count integer default 20
)
returns table (passage_id text, similarity double precision)
language sql stable security definer set search_path = public
as $$
  select rc.passage_id, 1 - (rc.embedding <=> query_embedding) as similarity
  from public.regulation_chunks rc
  order by rc.embedding <=> query_embedding
  limit greatest(match_count,1);
$$;
'''
(DEPLOY/"supabase_schema.sql").write_text(sql,encoding="utf-8")

1174

In [12]:
def sha256(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

version = {
    "artifact_name":"school-regulation-hybrid-retriever-e5",
    "created_at_utc":datetime.now(timezone.utc).isoformat(),
    "base_model":"intfloat/multilingual-e5-base",
    "corpus_rows":int(len(CORPUS)),
    "embedding_dimension":768,
    "corpus_field":FIELD,
}
with (DEPLOY/"version.json").open("w",encoding="utf-8") as f:
    json.dump(version,f,ensure_ascii=False,indent=2)

entries=[]
for p in sorted(x for x in DEPLOY.rglob("*") if x.is_file()):
    entries.append({"path":p.relative_to(DEPLOY).as_posix(),"size_bytes":p.stat().st_size,"sha256":sha256(p)})
with (DEPLOY/"manifest.json").open("w",encoding="utf-8") as f:
    json.dump({"artifact":version,"files":entries},f,ensure_ascii=False,indent=2)

In [13]:
zip_path = DEPLOY.parent/"hybrid_retriever_e5_deploy.zip"
if zip_path.exists(): zip_path.unlink()

with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in DEPLOY.rglob("*"):
        if p.is_file():
            z.write(p,Path("hybrid_retriever_e5")/p.relative_to(DEPLOY))

print("Created:",zip_path)

Created: /content/drive/MyDrive/UIT_Legal_System/artifacts/deploy/hybrid_retriever_e5_deploy.zip
